# Confidence Estimation: Single Document

How to use `compare_with(add_confidence_metrics=True)` to evaluate whether a model's confidence scores track correctness.

In [ ]:
from typing import Optional

from stickler.comparators import ExactComparator, LevenshteinComparator, NumericComparator
from stickler.structured_object_evaluator.models.comparable_field import ComparableField
from stickler.structured_object_evaluator.models.structured_model import StructuredModel

## 1. Define Model & Data

In [ ]:
class Invoice(StructuredModel):
    invoice_number: str = ComparableField(comparator=ExactComparator(), threshold=1.0, weight=3.0)
    vendor: str = ComparableField(comparator=LevenshteinComparator(), threshold=0.7, weight=1.0)
    total: float = ComparableField(comparator=NumericComparator(tolerance=0.01), weight=2.0)
    notes: Optional[str] = ComparableField(comparator=LevenshteinComparator(), threshold=0.5, weight=0.5)

gt = Invoice(
    invoice_number="INV-2024-001",
    vendor="Acme Corporation",
    total=1247.50,
    notes="Delivered to front door"
)

## 2. Confidence Structures in JSON

`from_json()` automatically unwraps `{"value": ..., "confidence": ...}` structures. Plain values work too.

In [ ]:
pred = Invoice.from_json({
    "invoice_number": {"value": "INV-2024-001", "confidence": 0.93},
    "vendor": {"value": "Acme Corp", "confidence": 0.72},
    "total": {"value": 1247.50, "confidence": 0.88},
    "notes": {"value": "Left at back entrance", "confidence": 0.45},
})

print(f"Values:       {pred.invoice_number}, {pred.vendor}, {pred.total}, {pred.notes}")
print(f"Confidences:  {pred.get_all_confidences()}")

## 3. compare_with + confidence_metrics

Pass `add_confidence_metrics=True` and `document_field_comparisons=True`. The result includes a `confidence_metrics` dict with overall and per-field metrics.

In [ ]:
result = gt.compare_with(
    pred,
    add_confidence_metrics=True,
    document_field_comparisons=True
)

print(f"Overall Score: {result['overall_score']:.3f}")
print(f"\nconfidence_metrics keys: {list(result['confidence_metrics'].keys())}")
print(f"\nOverall metrics:")
for name, val in result['confidence_metrics']['overall'].items():
    print(f"  {name}: {val}")

print(f"\nPer-field metrics:")
for field, metrics in result['confidence_metrics']['fields'].items():
    print(f"  {field}: {metrics}")

## 4. Inspecting the Join: field_comparisons + confidences

The confidence calculator joins `field_comparisons` (match/no-match per field) with `get_all_confidences()` (confidence per field path). Let's see both sides.

In [ ]:
confidences = pred.get_all_confidences()

print(f"{'Field':<20} {'Match':<8} {'Score':<8} {'Confidence':<12} {'Reason'}")
print("-" * 75)
for fc in result['field_comparisons']:
    field = fc['actual_key']
    match = '\u2713' if fc['match'] else '\u2717'
    score = f"{fc['score']:.3f}"
    conf = confidences.get(field)
    conf_str = f"{conf:.2f}" if conf is not None else '\u2014'
    print(f"{field:<20} {match:<8} {score:<8} {conf_str:<12} {fc['reason']}")

## 5. Nested Objects

Confidence paths use dot notation for nested fields.

In [ ]:
class Address(StructuredModel):
    street: str = ComparableField(comparator=LevenshteinComparator(), threshold=0.7)
    city: str = ComparableField(comparator=LevenshteinComparator(), threshold=0.7)
    zip_code: str = ComparableField(comparator=ExactComparator(), threshold=1.0)

class Customer(StructuredModel):
    name: str = ComparableField(comparator=LevenshteinComparator(), threshold=0.8)
    address: Address = ComparableField()

gt_cust = Customer(
    name="Jane Smith",
    address=Address(street="456 Oak Ave", city="Boston", zip_code="02101")
)
pred_cust = Customer.from_json({
    "name": {"value": "Jane Smith", "confidence": 0.96},
    "address": {
        "street": {"value": "456 Oak Avenue", "confidence": 0.85},
        "city": {"value": "Chicago", "confidence": 0.40},
        "zip_code": {"value": "02101", "confidence": 0.93}
    }
})

r = gt_cust.compare_with(pred_cust, add_confidence_metrics=True, document_field_comparisons=True)

print("Confidence paths:", list(pred_cust.get_all_confidences().keys()))
print("Field comparison paths:", [fc['actual_key'] for fc in r['field_comparisons']])
print()
print("Overall:", r['confidence_metrics']['overall'])
print("\nPer-field:")
for field, m in r['confidence_metrics']['fields'].items():
    print(f"  {field}: {m}")

## 6. Edge Cases

In [ ]:
# No confidence data — confidence_metrics not in result
pred_plain = Invoice(invoice_number="INV-2024-001", vendor="Acme Corporation", total=1247.50)
r_plain = gt.compare_with(pred_plain, add_confidence_metrics=True, document_field_comparisons=True)
print(f"No confidence data: 'confidence_metrics' in result = {'confidence_metrics' in r_plain}")
# When all fields have confidence but all match (single class), AUROC is None
pred_all_match = Invoice.from_json({
    "invoice_number": {"value": "INV-2024-001", "confidence": 0.95},
    "vendor": {"value": "Acme Corporation", "confidence": 0.90},
    "total": {"value": 1247.50, "confidence": 0.88},
    "notes": {"value": "Delivered to front door", "confidence": 0.80},
})
r_all = gt.compare_with(pred_all_match, add_confidence_metrics=True, document_field_comparisons=True)
print(f"All match, AUROC: {r_all['confidence_metrics']['overall']['auroc']}")

## 7. Partial Confidence Coverage

In [ ]:
pred_partial = Invoice.from_json({
    "invoice_number": {"value": "INV-2024-001", "confidence": 0.97},
    "vendor": "Acme Corporation",  # no confidence
    "total": {"value": 1247.50, "confidence": 0.91},
    "notes": {"value": "Wrong notes", "confidence": 0.25},
})
r_partial = gt.compare_with(pred_partial, add_confidence_metrics=True, document_field_comparisons=True)

print(f"Fields with confidence: {list(pred_partial.get_all_confidences().keys())}")
print(f"Fields in confidence_metrics: {list(r_partial['confidence_metrics']['fields'].keys())}")
print("\n'vendor' is excluded — no confidence score to evaluate.")